# HiMoDiT — TrainingTrains all four stages on a SMILES dataset with property columns.| Stage | Predicts | Capacity | Epochs | Notes ||---|---|---|---|---|| A1 | ring layout | 3M | 40 | ring types, fusion, linkers, spiro || A3 | branch topology | 10M | 100 | side-chain trees; the long one || A2 | atom identities | 10M | 40 | conditioned on the decoded bond graph || Terminal | decoration | 9M | 40 | 22 functional groups |Stages must run in this order — each conditions on what the previous oneproduced.**Interrupted sessions are fine.** Every stage auto-resumes from`latest.pt`, so re-running a cell picks up where it left off. The fullsequence takes longer than a single hosted session, so expect to run thisnotebook across two or three sittings. To restart a stage from scratch,delete its checkpoint directory.

## 1. Environment

In [ ]:
!pip install -q rdkit torch tqdm matplotlib

In [ ]:
import os, sys# On Colab, mount Drive and point REPO at the cloned repository.try:    from google.colab import drive    drive.mount('/content/drive', force_remount=False)    REPO = '/content/drive/MyDrive/himodit'    DATA = '/content/drive/MyDrive/himodit-data'except ImportError:    REPO = os.path.abspath('..')    DATA = os.path.join(REPO, 'data')assert os.path.isdir(REPO), f'repository not found at {REPO}'if REPO not in sys.path:    sys.path.insert(0, REPO)os.makedirs(DATA, exist_ok=True)CSV        = f'{DATA}/250k_rndm_zinc_drugs_clean_3.csv'LABELS     = f'{DATA}/labels.pkl'CKPT_ROOT  = f'{DATA}/checkpoints'PROPERTIES = ['logP', 'SAS']# Set True to reject molecules the encoder cannot represent without# dropping atoms. Costs ~6 points of retention, guarantees exact# round trips. See docs/limitations.md.STRICT = Falsefor name, path in [('repo', REPO), ('data', DATA), ('csv', CSV)]:    print(f'{name:6s} {path}')import torchprint(f'\ntorch {torch.__version__}  cuda={torch.cuda.is_available()}')if torch.cuda.is_available():    print(f'  {torch.cuda.get_device_name(0)}')

## 2. PreprocessEncodes every molecule into a layout label. Skipped if the label filealready exists — delete it to rebuild.Expected retention on ZINC250K is about 93%. A much lower number usuallymeans a capacity constant is too small for your dataset rather than thatthe molecules are unusable; check the rejection breakdown.

In [ ]:
import pickle, timefrom collections import Counterimport numpy as np, pandas as pdfrom himodit.chem.encoder import extract_layoutif os.path.exists(LABELS):    with open(LABELS, 'rb') as f:        labels = pickle.load(f)    print(f'reusing {LABELS} ({len(labels):,} labels)')else:    df = pd.read_csv(CSV)    df['smiles'] = df['smiles'].astype(str).str.strip()    df = df.dropna(subset=PROPERTIES)    print(f'{len(df):,} molecules with complete properties')    # Z-score the conditioning axes. Keep these statistics: evaluation    # needs the same ones to interpret targets.    stats, cond = {}, np.zeros((len(df), len(PROPERTIES)), dtype=np.float32)    for i, col in enumerate(PROPERTIES):        v = df[col].to_numpy(dtype=np.float64)        stats[col] = {'mean': float(v.mean()), 'std': float(v.std())}        cond[:, i] = (v - v.mean()) / v.std()        print(f"  {col}: mean={stats[col]['mean']:.4f} std={stats[col]['std']:.4f}")    labels, rejections, t0 = [], Counter(), time.time()    for idx, smi in enumerate(df['smiles'].tolist()):        label, reason = extract_layout(smi, strict=STRICT)        if label is None:            rejections[reason] += 1            continue        label['condition'] = cond[idx].copy()        labels.append(label)        if (idx + 1) % 20000 == 0:            rate = (idx + 1) / (time.time() - t0)            print(f'  {idx+1:>7,}  kept {len(labels):>7,}  '                  f'retention {100*len(labels)/(idx+1):5.2f}%  '                  f'eta {(len(df)-idx-1)/rate/60:.1f} min')    print(f'\nkept {len(labels):,}/{len(df):,} '          f'({100*len(labels)/len(df):.2f}%) in {(time.time()-t0)/60:.1f} min')    for reason, n in rejections.most_common(10):        print(f'  {n:>7,}  {reason}')    with open(LABELS, 'wb') as f:        pickle.dump(labels, f, protocol=4)    print(f'wrote {LABELS}')

## 3. Check the label schemaConfirms the labels carry every field the four stages read, and that thecapacity constants agree across encoder, models, and decoder. A mismatchhere surfaces later as an opaque shape error deep in training.

In [ ]:
from himodit.chem.decoder import R_MAX, M_MAX, L_MAX, B_LEN_MAX, P_MAX_BRANCHfrom himodit.models.ring_layout import N_R_CLASSES, N_F_CLASSES, N_SPIRO_POS_CLASSESfrom himodit.models.branch_topology import P_MAX as A3_P_MAXfrom himodit.models.ring_atom import N_ATOM_CLASSESprint(f'R_MAX={R_MAX}  M_MAX={M_MAX}  L_MAX={L_MAX}  '      f'B_LEN_MAX={B_LEN_MAX}  P_MAX_BRANCH={P_MAX_BRANCH}')print(f'A1 vocab: R={N_R_CLASSES} F={N_F_CLASSES} spiro={N_SPIRO_POS_CLASSES}')print(f'A2 atom vocab: {N_ATOM_CLASSES}')assert A3_P_MAX == P_MAX_BRANCH, 'A3 branch slots disagree with the encoder'required = ('R', 'F', 'L', 'B_size', 'B_pos', 'B_parent', 'B_bond',            'spiro_atom_positions', 'atom_ids', 'M_total', 'terminals',            'condition')missing = [k for k in required if k not in labels[0]]assert not missing, f'labels are missing {missing}; re-run preprocessing'print(f'\n{len(labels):,} labels, all required fields present')print(f'condition dim: {len(labels[0]["condition"])}')print(f'mean scaffold size: '      f'{np.mean([l["M_total"] for l in labels]):.1f} atoms')

---## 4. Stage A1 — ring layoutRoughly 2 hours at capacity 3M. The per-epoch `decode` figure is thefraction of sampled layouts the decoder accepts; it should climb throughtraining and is the earliest signal that A1 is learning valid topology.

In [ ]:
from himodit.training.a1 import train_a1train_a1(    labels_pkl_path=LABELS,    ckpt_dir=f'{CKPT_ROOT}/a1',    num_epochs=40,    batch_size=256,    capacity='3M',    lr=3e-4,    cfg_drop_prob=0.10,)

---## 5. Stage A3 — branch topologyThe longest stage, roughly 10 hours at capacity 10M. It will not finishin one hosted session — re-run this cell after a disconnect and itresumes.A higher condition-dropout probability (0.30) is used here becausebranch topology is only weakly determined by the property target, and astronger unconditional signal gives classifier-free guidance something toextrapolate from.

In [ ]:
from himodit.training.a3 import train_a3train_a3(    labels_pkl_path=LABELS,    ckpt_dir=f'{CKPT_ROOT}/a3',    num_epochs=100,    batch_size=256,    capacity='10M',    lr=3e-4,    cfg_drop_prob=0.30,)

---## 6. Stage A2 — atom identitiesAbout an hour. A2 sees the decoded bond graph as a clean input, so itsloss falls quickly.

In [ ]:
from himodit.training.a2 import train_a2train_a2(    labels_pkl_path=LABELS,    ckpt_dir=f'{CKPT_ROOT}/a2',    num_epochs=40,    batch_size=256,    capacity='10M',    lr=3e-4,    cfg_drop_prob=0.10,)

---## 7. Terminal — decorationAbout 90 minutes. Class weighting is on by default: the fragmentdistribution is heavily skewed toward a few common groups, and without itthe rare terminals are never predicted.

In [ ]:
from himodit.training.terminal import train_terminaltrain_terminal(    labels_pkl_path=LABELS,    ckpt_dir=f'{CKPT_ROOT}/terminal',    num_epochs=40,    batch_size=256,    capacity='9M',    lr=3e-4,    cfg_drop_prob=0.10,)

---## 8. Training curves

In [ ]:
import jsonimport matplotlib.pyplot as pltstages = ['a1', 'a3', 'a2', 'terminal']fig, axes = plt.subplots(1, len(stages), figsize=(4.5 * len(stages), 3.5))for ax, stage in zip(axes, stages):    path = f'{CKPT_ROOT}/{stage}/history.json'    if not os.path.exists(path):        ax.set_title(f'{stage}: not trained yet')        ax.axis('off')        continue    with open(path) as f:        history = json.load(f)    epochs = [h['epoch'] for h in history]    ax.plot(epochs, [h['train_loss'] for h in history], label='train')    ax.plot(epochs, [h['val_loss'] for h in history], label='val')    ax.set_title(f'{stage} ({len(history)} epochs)')    ax.set_xlabel('epoch'); ax.set_ylabel('loss')    ax.grid(alpha=0.3); ax.legend()plt.tight_layout(); plt.show()

## DoneAll four checkpoint directories now hold `best_model.pt`. Open`02_evaluate.ipynb` to generate molecules and measure V·U·N andcontrollability.